# OceanScan - Model Training (YOLOv8n)

Trains a **YOLOv8n** nano object detector on the **noise-filtered** sonar images in `output/noise_filter/train_noise_filter/`.

- **Train set**: `output/noise_filter/train_noise_filter/train` (9600 images)
- **Val set**:   `output/noise_filter/train_noise_filter/val` (984 images)
- **Classes**:   `0 pipe`, `1 ship`, `2 human`, `3 crabpot`, `4 plane`, `5 cylinder` (YOLO format)
- **Early stopping**: `patience=25` -> stops automatically when val mAP plateaus (anti-overfit)
- **Device**: auto-detects Intel Arc GPU via torch-directml (`dml`), falls back to CPU

At the end, the best weights are saved to `backend/best.pt` for the prediction notebook.

## 0. Dependencies

"Run All" auto-installs anything that is missing (e.g. `cv2`), so no manual `pip install` is needed.

In [ ]:
import importlib
import subprocess
import sys

print("Kernel Python:", sys.executable)

_DEP_MAP = {
    "numpy": "numpy",
    "cv2": "opencv-python",
    "matplotlib": "matplotlib",
    "pandas": "pandas",
    "tqdm": "tqdm",
    "ultralytics": "ultralytics",
}

missing = []
for _mod, _pkg in _DEP_MAP.items():
    try:
        importlib.import_module(_mod)
    except ImportError:
        missing.append(_pkg)

if missing:
    print("Installing missing packages:", ", ".join(sorted(missing)))
    subprocess.check_call([sys.executable, "-m", "pip", "install", *sorted(missing)])

failed = []
for _mod in _DEP_MAP:
    try:
        importlib.import_module(_mod)
    except ImportError:
        failed.append(_mod)

if failed:
    print("!!! STILL MISSING:", failed)
    print("Open a terminal here and run:  pip install -r requirements.txt")
else:
    print("All dependencies OK.")

## 1. Setup

In [ ]:
import shutil
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from ultralytics import YOLO

ROOT = Path(r"D:\1. Project Program\1.SIH\OceanScan")
OUT = ROOT / "output" / "noise_filter" / "train_noise_filter"
DATA_YAML = OUT / "data.yaml"
SUBSETS = ["train", "val"]

names_block = DATA_YAML.read_text().split("names:")[-1].strip().splitlines()
NAMES = {}
for _line in names_block:
    _k, _v = _line.split(":", 1)
    NAMES[int(_k)] = _v.strip().strip('"')
print("Classes:", NAMES)

# --- Detect best device for training (Intel Arc via DirectML, else CPU) ---
try:
    import torch_directml as _dml
    _dev = _dml.device()
    _dev_name = _dml.device_name(0)
    print(f"GPU found via torch-directml: {_dev_name}")
    DEVICE = _dev  # torch.device('privateuseone:0'); ultralytics rejects the string 'dml'
except Exception as _e:
    print(f"torch-directml not available ({_e}); using CPU.")
    DEVICE = "cpu"
print("Training device:", DEVICE)
# --- DirectML compatibility patches for ultralytics 8.4.x (assumes CUDA-style backend) ---
if DEVICE != "cpu":
    import sys as _sys
    import torch as _torch
    from types import SimpleNamespace as _NS

    _Mod = _torch.get_device_module("privateuseone")
    for _n in ("empty_cache", "synchronize", "memory_allocated", "set_device"):
        if not hasattr(_Mod, _n):
            setattr(_Mod, _n, staticmethod(lambda *a, **k: None))
    if not hasattr(_Mod, "memory_reserved"):
        setattr(_Mod, "memory_reserved", staticmethod(lambda *a, **k: 0))
    if not hasattr(_Mod, "get_device_properties"):
        setattr(_Mod, "get_device_properties", staticmethod(lambda *a, **k: _NS(total_memory=0)))

    import ultralytics.utils.torch_utils as _tu
    _orig_sel = _tu.select_device

    def _dml_select(device="", newline=False, verbose=True):
        if isinstance(device, _torch.device) and device.type == "privateuseone":
            return device
        if isinstance(device, str) and device.startswith("privateuseone"):
            return _torch.device(device)
        return _orig_sel(device, newline, verbose)

    _tu.select_device = _dml_select
    for _m in list(_sys.modules.values()):
        if getattr(_m, "__name__", "").startswith("ultralytics") and hasattr(_m, "select_device"):
            setattr(_m, "select_device", _dml_select)

    # --- Additional DirectML compatibility patches (verified by GPU smoke test) ---
    # v8DetectionLoss.preprocess uses unique(return_counts=True), which this DirectML
    # build does not implement -> assemble the (small) targets tensor on CPU.
    from ultralytics.utils.loss import v8DetectionLoss as _v8loss
    from ultralytics.utils.ops import xywh2xyxy as _xywh2xyxy

    def _preprocess_cpu(self, targets, batch_size, scale_tensor):
        targets = targets.cpu()
        scale_tensor = scale_tensor.cpu()
        nl, ne = targets.shape
        if nl == 0:
            out = _torch.zeros(batch_size, 0, ne - 1)
        else:
            batch_idx = targets[:, 0].long()
            _, counts = batch_idx.unique(return_counts=True)
            counts = counts.to(dtype=_torch.int32)
            out = _torch.zeros(batch_size, counts.max(), ne - 1)
            offsets = _torch.zeros(batch_size + 1, dtype=_torch.long)
            offsets.scatter_add_(0, batch_idx + 1, _torch.ones_like(batch_idx))
            offsets = offsets.cumsum(0)
            within_idx = _torch.arange(nl) - offsets[batch_idx]
            out[batch_idx, within_idx] = targets[:, 1:]
            out[..., 1:5] = _xywh2xyxy(out[..., 1:5].mul_(scale_tensor))
        return out.to(self.device)

    _v8loss.preprocess = _preprocess_cpu

    # Task-aligned label assigner uses topk / nonzero / scatter_add_(dupe indices) /
    # bbox_iou ops that DirectML lacks -> run the whole (small) assigner on CPU.
    from ultralytics.utils.tal import TaskAlignedAssigner as _tal

    _orig_tal_forward = _tal._forward

    def _tal_cpu(self, *args):
        _dev = args[0].device
        outs = _orig_tal_forward(self, *[a.cpu() for a in args])
        return [o.to(_dev) for o in outs]

    _tal._forward = _tal_cpu

    # DirectML cannot run ops under torch.inference_mode() (ultralytics'
    # @smart_inference_mode on validation) -> "Cannot set version_counter for
    # inference tensor". Re-wrap validation with torch.no_grad().
    from ultralytics.engine.validator import BaseValidator as _BV

    _orig_val_call = _BV.__call__.__wrapped__

    def _val_call_no_grad(self, *args, **kwargs):
        with _torch.no_grad():
            return _orig_val_call(self, *args, **kwargs)

    _BV.__call__ = _val_call_no_grad

    # DirectML throws E_INVALIDARG in some NMS ops -> run NMS on CPU.
    import ultralytics.utils.nms as _nms

    _orig_nms = _nms.non_max_suppression

    def _nms_cpu(prediction, *args, **kwargs):
        if isinstance(prediction, (tuple, list)):
            prediction = prediction[0]
        _dev = prediction.device
        out = _orig_nms(prediction.cpu(), *args, **kwargs)
        if isinstance(out, tuple):
            return tuple(o.to(_dev) if isinstance(o, _torch.Tensor) else o for o in out)
        return [o.to(_dev) if isinstance(o, _torch.Tensor) else o for o in out]

    _nms.non_max_suppression = _nms_cpu

    # metrics.box_iou also hits E_INVALIDARG (min/max/chunk/prod) -> run on CPU.
    import ultralytics.utils.metrics as _met

    _orig_box_iou = _met.box_iou

    def _box_iou_cpu(box1, box2):
        _dev = box1.device
        a = box1.cpu() if box1.device.type != "cpu" else box1
        b = box2.cpu() if box2.device.type != "cpu" else box2
        out = _orig_box_iou(a, b)
        return out.to(_dev) if _dev.type != "cpu" else out

    _met.box_iou = _box_iou_cpu

    # This DirectML build cannot run fp16 AMP -> disable AMP for stable DirectML training
    AMP = False
else:
    AMP = True


In [ ]:
print(DATA_YAML.read_text())
for sub in SUBSETS:
    n_img = len(list((OUT / sub / "images").glob("*")))
    n_lbl = len(list((OUT / sub / "labels").glob("*.txt")))
    print(f"{sub:35s} images={n_img}  labels={n_lbl}")

## 2. Train YOLOv8n with early stopping

Run `shutil.rmtree` on `backend/runs/dataset_yolov8` first to retrain from scratch.

In [ ]:
RUNS = ROOT / "backend" / "runs" / "dataset_yolov8"
# Delete a previous run to retrain from scratch:
shutil.rmtree(RUNS, ignore_errors=True)

# Choose model size. nano is fast/low-memory; switch to yolov8s/m for more accuracy if VRAM/CPU allows.
MODEL_SIZE = "n"   # 'n' nano | 's' small | 'm' medium
# If DirectML training fails, retry on CPU automatically.

model = YOLO(f"yolov8{MODEL_SIZE}.pt")  # downloads pretrained weights on first run

def _train(device):
    return model.train(
        data=str(DATA_YAML),
        epochs=80,           # generous budget; early stopping below stops it when converged
        patience=25,         # auto early-stop if val mAP stalls for 25 epochs (anti-overfit)
        imgsz=640,           # input size - good balance of speed & accuracy
        batch=8,             # small batch fits 2GB VRAM / CPU
        device=device,       # torch.device('privateuseone:0') or 'cpu'
        workers=0,
        amp=AMP,  # DirectML build has no fp16 AMP support; CPU keeps AMP
        project=str(ROOT / "backend" / "runs"),
        name="dataset_yolov8",
        exist_ok=True,
        cos_lr=True,         # cosine LR schedule -> better convergence, less overfitting
        close_mosaic=10,     # disable mosaic last 10 epochs -> cleaner val estimates
        verbose=True,
    )

try:
    print("Starting training on", DEVICE)
    results = _train(DEVICE)
except Exception as e:
    if str(DEVICE).startswith("privateuseone"):
        print(f"DirectML training failed ({e}). Falling back to CPU...")
        DEVICE = "cpu"
        results = _train("cpu")
    else:
        raise e

## 3. Training curves

In [ ]:
csv_path = RUNS / "results.csv"
if not csv_path.exists():
    print("No results.csv yet - run the training cell above first.")
else:
    df = pd.read_csv(csv_path)
    cols = set(df.columns)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    if "metrics/mAP50(B)" in cols:
        axes[0].plot(df["epoch"], df["metrics/mAP50(B)"], label="val mAP50")
    if "metrics/mAP50-95(B)" in cols:
        axes[0].plot(df["epoch"], df["metrics/mAP50-95(B)"], label="val mAP50-95")
    axes[0].set(xlabel="epoch", title="Val metrics")
    if axes[0].lines:
        axes[0].legend()
    axes[0].grid(alpha=.3)
    if "train/box_loss" in cols:
        axes[1].plot(df["epoch"], df["train/box_loss"], label="train box")
    if "val/box_loss" in cols:
        axes[1].plot(df["epoch"], df["val/box_loss"], label="val box")
    axes[1].set(xlabel="epoch", title="Box loss")
    if axes[1].lines:
        axes[1].legend()
    axes[1].grid(alpha=.3)
    plt.tight_layout(); plt.show()

## 4. Evaluate best weights on the validation split

In [ ]:
best = RUNS / "weights" / "best.pt"
print("best.pt exists:", best.exists())
if best.exists():
    m = YOLO(str(best)).val(split="val", verbose=False)
    print(f"mAP50     : {m.box.map50:.4f}")
    print(f"mAP50-95  : {m.box.map:.4f}")
    for c in sorted(NAMES):
        print(f"  {NAMES[c]:6s} mAP50 = {m.box.maps[c]:.4f}")

## 5. Save model for the prediction notebook

In [ ]:
if best.exists():
    target = ROOT / "backend" / "best.pt"
    shutil.copy2(str(best), str(target))
    print(f"Saved -> {target}")
else:
    print("Training did not complete - no best.pt to save.")